# 18-phase18-large-long

**neuron Phase 18** — Phase 17 의 핵심 한계 (**plateau 미도달**) 해소 + **larger model + longer training** 동시 적용.

**배경**: Phase 17 sweep (~6M params, 5000 step) 의 핵심 발견:
- scale-up 의 절대적 효과 명확 (loss 0.5-0.6 ↓, ppl 6.7 → 3.8)
- dynamic method advantage 부분 진전 (grown 격차 +0.043 → +0.026 축소)
- **plateau 미도달** — 모든 mode 가 drift -0.006 (still learning)

→ **5000 step 도 부족**. Phase 18 은 **larger model (2x) + longer training (3x)** 동시 적용으로 *진정한 plateau* 도달 시도.

**핵심 가설**:
1. **plateau 도달** — 충분한 학습 시간으로 진정한 final loss 측정 (drift < -0.001)
2. **DST advantage 발현** — 더 큰 model + 긴 학습으로 swap cost amortize 가능?
3. **grown ≈ dense** — 더 긴 post-grow 학습 (12500 step) 으로 0-init activation 충분?
4. **Phase 17 → 18 trend** — dynamic advantage 가설의 추가 진전?

설계: 5 mode × 2 seed = 10 run at **large model + long training**.

| 항목 | Phase 17 | Phase 18 | 배수 |
|---|---|---|---|
| hidden_dim | 256 | **512** | 2x |
| n_heads | 8 | **16** (head_dim=32 유지) | 2x |
| n_layers | 6 | **12** | 2x |
| ffn_dim (large) | 512 | **1024** | 2x |
| max_steps | 5000 | **15000** | 3x |
| approx params | ~6.4M | **~30-40M** | ~5-6x |

5 mode (Phase 17 와 동일 구성, hp 만 scale 맞춤):
- `dense_large`: 큰 모델 baseline (target reference)
- `static_prune_50`: dense 학습 후 step 7500 에서 50% prune
- `DST_RigL_p16hp`: Phase 16a 동일 hp (period=50, swap=0.1) — 순수 scale 효과 분리 control
- `DST_RigL_p17hp`: Phase 17 보수적 hp (period=200, swap=0.05) — scale + hp 효과
- `grown`: ffn=512 시작 → step 7500 에서 1024 로 grow

arch 고정: `hybrid_around_one_around_one` + `use_full_graph=True`
데이터: TinyShakespeare (char-LM, block_size=64 유지 — vocab/data 동일, scale 만 model + train)
시드: [42, 123]
Compute 추정: 단일 GPU 4-6 시간
작성일: 2026-05-29
연관: Issue [#82](https://github.com/EinSofINTEREST/GraphLM/issues/82) / Phase 17 PR [#81](https://github.com/EinSofINTEREST/GraphLM/pull/81)

## 0. 환경 / 의존성

In [ ]:
from __future__ import annotations

import math
import statistics
from pathlib import Path

import matplotlib.pyplot as plt
import torch

from graphlm.data.tinyshakespeare import (
    CharTokenizer,
    TinyShakespeareDataset,
    load_tinyshakespeare_text,
)
from graphlm.neuron.hybrid_transformer_demo import (
    HybridTransformerTrainConfig,
    train_hybrid_transformer_lm,
)
from graphlm.utils import safe_perplexity

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}")
print(f"torch: {torch.__version__}")
if device == "cpu":
    print("⚠️  Phase 18 은 ~30-40M params + 15K step — GPU 강력 권장")

## 1. Config — large model + long training

In [ ]:
text = load_tinyshakespeare_text()
tokenizer = CharTokenizer(text)
dataset = TinyShakespeareDataset(text, tokenizer)
vocab_size = tokenizer.vocab_size
print(f"vocab_size = {vocab_size}, dataset size = {len(dataset)}")

# Phase 18: large model (2x Phase 17) + long training (3x Phase 17)
HIDDEN_DIM = 512
N_HEADS = 16  # head_dim = 32 (512/16)
N_LAYERS = 12
GROUP_SIZE = 16
BLOCK_SIZE = 64
BATCH_SIZE = 32
LR = 3e-4
MAX_STEPS = 15000
DYNAMIC_AT_STEP = MAX_STEPS // 2  # 7500
# Phase 16a hyperparameter (control — scale 효과 only 분리)
DST_PERIOD_P16 = 50
DST_SWAP_FRACTION_P16 = 0.1
# Phase 17 보수적 hyperparameter
DST_PERIOD_P17 = 200
DST_SWAP_FRACTION_P17 = 0.05
# 마지막 1500 step stabilize (Phase 17 의 500 의 3x)
DST_END_STEP = max(1, MAX_STEPS - 1500)
SEEDS = [42, 123]
ARCH = "hybrid_around_one_around_one"
FFN_SMALL = 512
FFN_LARGE = 1024

MODES = [
    {
        "name": "dense_large",
        "ffn_dim": FFN_LARGE,
        "prune_at_step": None,
        "prune_fraction": 0.0,
        "regrow_method": None,
        "dst_period": None,
        "dst_swap_fraction": 0.0,
        "grow_at_step": None,
        "grow_ffn_target": None,
    },
    {
        "name": "static_prune_50",
        "ffn_dim": FFN_LARGE,
        "prune_at_step": DYNAMIC_AT_STEP,
        "prune_fraction": 0.5,
        "regrow_method": None,
        "dst_period": None,
        "dst_swap_fraction": 0.0,
        "grow_at_step": None,
        "grow_ffn_target": None,
    },
    {
        "name": "DST_RigL_p16hp",
        "ffn_dim": FFN_LARGE,
        "prune_at_step": DYNAMIC_AT_STEP,
        "prune_fraction": 0.5,
        "regrow_method": "rigl",
        "dst_period": DST_PERIOD_P16,
        "dst_swap_fraction": DST_SWAP_FRACTION_P16,
        "grow_at_step": None,
        "grow_ffn_target": None,
    },
    {
        "name": "DST_RigL_p17hp",
        "ffn_dim": FFN_LARGE,
        "prune_at_step": DYNAMIC_AT_STEP,
        "prune_fraction": 0.5,
        "regrow_method": "rigl",
        "dst_period": DST_PERIOD_P17,
        "dst_swap_fraction": DST_SWAP_FRACTION_P17,
        "grow_at_step": None,
        "grow_ffn_target": None,
    },
    {
        "name": "grown",
        "ffn_dim": FFN_SMALL,
        "prune_at_step": None,
        "prune_fraction": 0.0,
        "regrow_method": None,
        "dst_period": None,
        "dst_swap_fraction": 0.0,
        "grow_at_step": DYNAMIC_AT_STEP,
        "grow_ffn_target": FFN_LARGE,
    },
]
print("\nPhase 18 config (large + long):")
print(
    f"  hidden={HIDDEN_DIM}, n_heads={N_HEADS}, n_layers={N_LAYERS}, ffn(L)={FFN_LARGE}, group={GROUP_SIZE}"
)
print(f"  max_steps={MAX_STEPS}, dynamic_at={DYNAMIC_AT_STEP}, dst_end={DST_END_STEP}")
print(f"  DST p16hp: period={DST_PERIOD_P16}, swap={DST_SWAP_FRACTION_P16}")
print(f"  DST p17hp: period={DST_PERIOD_P17}, swap={DST_SWAP_FRACTION_P17}")
print(f"  post-dynamic 학습 시간: {DYNAMIC_AT_STEP} step (Phase 17 의 {2500} 의 3x)")

## 2. Sweep 실행 (5 mode × 2 seed = 10 run, GPU 강력 권장)

추정 시간: 단일 GPU 4-6 시간. 각 run ~ 25-35 분 (30M params × 15K step).

In [ ]:
results = {}
for mode in MODES:
    for seed in SEEDS:
        key = (mode["name"], seed)
        print(f"\n== mode={mode['name']} seed={seed} ==")
        cfg = HybridTransformerTrainConfig(
            dataset=dataset,
            vocab_size=vocab_size,
            hidden_dim=HIDDEN_DIM,
            n_heads=N_HEADS,
            ffn_dim=mode["ffn_dim"],
            n_layers=N_LAYERS,
            group_size=GROUP_SIZE,
            arch=ARCH,
            use_full_graph=True,
            block_size=BLOCK_SIZE,
            batch_size=BATCH_SIZE,
            lr=LR,
            max_steps=MAX_STEPS,
            prune_at_step=mode["prune_at_step"],
            prune_fraction=mode["prune_fraction"],
            regrow_method=mode["regrow_method"],
            dst_period=mode["dst_period"],
            dst_swap_fraction=mode["dst_swap_fraction"],
            dst_end_step=DST_END_STEP if mode["regrow_method"] else None,
            grow_at_step=mode["grow_at_step"],
            grow_ffn_target=mode["grow_ffn_target"],
            seed=seed,
            device=device,
        )
        out = train_hybrid_transformer_lm(cfg)
        results[key] = out
        n_cycles = len(out["dst_cycles"])
        print(
            f"  final_loss = {out['final_loss']:.4f}  (ppl = {safe_perplexity(out['final_loss']):.2f})"
            f"  params = {out['final_param_count']:,}"
            f"  sparsity = {out['final_sparsity']:.3f}  dst_cycles = {n_cycles}"
        )

## 3. 결과 표 + 자동 verdict (plateau 도달 검증 추가)

In [ ]:
print(
    f"{'mode':>16s} {'seed':>6s} {'final_loss':>12s} {'perplexity':>12s} "
    f"{'params':>12s} {'sparsity':>10s}"
)
print("-" * 80)
for (name, seed), out in results.items():
    fl = out["final_loss"]
    print(
        f"{name:>16s} {seed:>6d} {fl:>12.4f} {safe_perplexity(fl):>12.2f} "
        f"{out['final_param_count']:>12,} {out['final_sparsity']:>10.3f}"
    )

# mode 별 평균
print("\n== Mode summary (mean ± σ across seeds) ==")
summary = {}
for mode in MODES:
    name = mode["name"]
    vals = [results[(name, s)]["final_loss"] for s in SEEDS]
    params = [results[(name, s)]["final_param_count"] for s in SEEDS]
    sparsities = [results[(name, s)]["final_sparsity"] for s in SEEDS]
    m = statistics.mean(vals)
    sd = statistics.stdev(vals) if len(vals) > 1 else 0.0
    summary[name] = (m, sd, statistics.mean(params), statistics.mean(sparsities))
    print(
        f"  {name:>16s} {m:.4f} ± {sd:.4f}  (ppl ≈ {safe_perplexity(m):.2f})  "
        f"params={statistics.mean(params):,.0f}  sparsity={statistics.mean(sparsities):.3f}"
    )

# 자동 verdict
print("\n== Verdict ==")
dense_loss = summary["dense_large"][0]
static_loss = summary["static_prune_50"][0]
rigl_p16_loss = summary["DST_RigL_p16hp"][0]
rigl_p17_loss = summary["DST_RigL_p17hp"][0]
grown_loss = summary["grown"][0]

all_finite = all(math.isfinite(out["final_loss"]) for out in results.values())
verdict_1 = "PASS" if all_finite else "FAIL"
print(f"1. all-finite: {all_finite}  [{verdict_1}]")

diff_rigl_static = rigl_p16_loss - static_loss
verdict_2 = "PASS" if diff_rigl_static <= 0.02 else "FAIL"
print(
    f"2. DST_RigL_p16hp ≤ static + 0.02 (순수 scale + long-train 효과): "
    f"diff = {diff_rigl_static:+.4f}  [{verdict_2}]"
)
if diff_rigl_static < 0:
    print("   → DST advantage 발현! Phase 16a 의 scale-limit 가설 입증.")
elif diff_rigl_static < 0.005:
    print("   → DST 가 static 근방까지 도달 — 부분 advantage.")
else:
    print("   → DST 여전히 static 보다 약간 열위.")
diff_hp = rigl_p17_loss - rigl_p16_loss
print(f"   2b. hp 효과: p17hp - p16hp = {diff_hp:+.4f}")

diff_grown_dense = grown_loss - dense_loss
verdict_3 = "PASS" if diff_grown_dense <= 0.02 else "FAIL"
print(
    f"3. grown ≤ dense_large + 0.02 (16b 재검증, long post-grow): "
    f"diff = {diff_grown_dense:+.4f}  [{verdict_3}]"
)

# 4. plateau 도달 검증 — Phase 18 의 핵심 verdict
print("\n4. plateau 도달 검증 (drift < -0.001 = true plateau):")
all_plateau = True
for name in [m["name"] for m in MODES]:
    drifts = []
    for seed in SEEDS:
        losses = results[(name, seed)]["losses"]
        last_500 = losses[-500:]
        half_len = len(last_500) // 2
        if half_len > 0:
            first_half = sum(last_500[:half_len]) / half_len
            second_half = sum(last_500[half_len:]) / (len(last_500) - half_len)
            drifts.append(second_half - first_half)
        else:
            drifts.append(0.0)
    mean_drift = sum(drifts) / len(drifts)
    plateau_reached = mean_drift > -0.001
    all_plateau = all_plateau and plateau_reached
    marker = "✓" if plateau_reached else "✗"
    print(f"     {name:>16s} mean drift = {mean_drift:+.4f}  {marker}")
verdict_4 = "PASS" if all_plateau else "FAIL"
print(f"   → all-mode plateau 도달: {all_plateau}  [{verdict_4}]")
if not all_plateau:
    print("   → 일부 mode 여전히 학습 중 — 본 sweep 도 수렴 영역 미달 가능성")

## 4. Loss curve 시각화 — 15K step 전체

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 6))
colors = {
    "dense_large": "tab:green",
    "static_prune_50": "tab:red",
    "DST_RigL_p16hp": "tab:purple",
    "DST_RigL_p17hp": "tab:blue",
    "grown": "tab:orange",
}
window = 200  # Phase 18 의 longer training 에 맞춰 window 도 키움

for mode in MODES:
    name = mode["name"]
    losses_per_seed = [results[(name, s)]["losses"] for s in SEEDS]
    smoothed = []
    for losses in losses_per_seed:
        smoothed.append(
            [
                sum(losses[max(0, i - window + 1) : i + 1]) / min(i + 1, window)
                for i in range(len(losses))
            ]
        )
    arr = torch.tensor(smoothed)
    mean = arr.mean(dim=0)
    std = arr.std(dim=0) if arr.size(0) > 1 else torch.zeros_like(mean)
    steps = list(range(len(mean)))
    ax.plot(steps, mean, label=name, color=colors[name], linewidth=1.5)
    ax.fill_between(steps, mean - std, mean + std, color=colors[name], alpha=0.12)

ax.axvline(
    DYNAMIC_AT_STEP,
    color="black",
    linestyle=":",
    alpha=0.5,
    label=f"prune/grow @ {DYNAMIC_AT_STEP}",
)
ax.axvline(DST_END_STEP, color="gray", linestyle=":", alpha=0.4, label=f"DST end @ {DST_END_STEP}")

ax.set_xlabel("step")
ax.set_ylabel(f"loss (rolling mean w={window})")
ax.set_title(
    "Phase 18 large+long (hidden=512, n_layers=12, ffn(L)=1024, steps=15000) — 5 mode (mean ± σ over 2 seeds)"
)
ax.legend(loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()

out_dir = Path("../../runs/notebook-neuron-phase18")
out_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(out_dir / "loss_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"saved: {out_dir / 'loss_curves.png'}")

## 5. Phase 16/17/18 3단계 progression 비교

In [ ]:
# Phase 16/17 baseline 수치 (Notion 정리 페이지 출처)
# Phase 16a: https://www.notion.so/36de8b70b7aa819aaa54d561fc222fa1
# Phase 16b: https://www.notion.so/36ee8b70b7aa811bb354c4fc98a484ed
# Phase 17:  https://www.notion.so/36fe8b70b7aa8168a61ce9fd27cf8625
p16 = {
    "dense": 1.8999,
    "static_prune_50": 1.9777,
    "DST_RigL_p16hp": 1.9831,  # 16a 의 DST_RigL_50 (period=50, swap=0.1)
    "grown": 1.9428,
}
p17 = {
    "dense": 1.3326,
    "static_prune_50": 1.3496,
    "DST_RigL_p16hp": 1.3588,
    "grown": 1.3582,
}
p18 = {
    "dense": summary["dense_large"][0],
    "static_prune_50": summary["static_prune_50"][0],
    "DST_RigL_p16hp": summary["DST_RigL_p16hp"][0],
    "grown": summary["grown"][0],
}

fig, ax = plt.subplots(1, 1, figsize=(11, 5))
modes_plot = ["dense", "static_prune_50", "DST_RigL_p16hp", "grown"]
x = list(range(len(modes_plot)))
width = 0.27
p16_vals = [p16[m] for m in modes_plot]
p17_vals = [p17[m] for m in modes_plot]
p18_vals = [p18[m] for m in modes_plot]
ax.bar([i - width for i in x], p16_vals, width, label="Phase 16 (~1M, 1.5K step)", color="#bcd5e6")
ax.bar(x, p17_vals, width, label="Phase 17 (~6M, 5K step)", color="#5b8db8")
ax.bar([i + width for i in x], p18_vals, width, label="Phase 18 (~30M, 15K step)", color="#1e3a5f")
ax.set_xticks(x)
ax.set_xticklabels(modes_plot, rotation=15)
ax.set_ylabel("final loss (mean over 2 seeds)")
ax.set_title("Phase 16 → 17 → 18 — scale progression (same hp)")
ax.legend()
ax.grid(alpha=0.3, axis="y")
plt.tight_layout()
fig.savefig(out_dir / "phase16_17_18_progression.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"saved: {out_dir / 'phase16_17_18_progression.png'}")

# 격차 변화 정량화
print("\n== dynamic advantage trend (Phase 16 → 17 → 18) ==")
for label, a_key, b_key in [
    ("DST - static", "DST_RigL_p16hp", "static_prune_50"),
    ("grown - dense", "grown", "dense"),
]:
    d16 = p16[a_key] - p16[b_key]
    d17 = p17[a_key] - p17[b_key]
    d18 = p18[a_key] - p18[b_key]
    print(f"  {label:>16s}: P16={d16:+.4f}  P17={d17:+.4f}  P18={d18:+.4f}")

## 6. 결론 / 다음 단계

(셀 출력 보고 사용자가 채울 영역)

**핵심 질문**:
1. **plateau 도달했나?** verdict 4 결과
2. **DST advantage 발현?** verdict 2 의 diff 가 음수 또는 0.005 이내?
3. **grown 이 dense 수준까지 도달?** verdict 3 의 diff 가 0.02 이내?
4. **Phase 16 → 17 → 18 의 trend**: dynamic advantage 의 progression?

**Phase 19 후보** (사용자 결정):
- (plateau + dynamic advantage 발현) → **결과 정리 + paper writing**
- (plateau but dynamic 여전히 미실현) → method 개선 (Net2WiderNet duplication, RigL scheduler) 또는 real dataset
- (plateau 미도달) → 더 긴 학습 또는 더 큰 model